# Projeto Final: Análise de Mobilidade e Transporte Metropolitano

**Aluno:** Felipe Alexandre Alves da Silva
**Disciplina:** Modelos Descritivos
**Professor:** Matheus Lacerda


---

## 📌 Objetivo
Este projeto tem como objetivo aplicar técnicas de **Aprendizado Não Supervisionado (Clusterização)** para analisar dados de mobilidade urbana em Regiões Metropolitanas brasileiras. 

A partir da base de dados **PEMOB 2025 (Pesquisa de Mobilidade Urbana)**, buscaremos agrupar as regiões com características operacionais e financeiras semelhantes, gerando insights estratégicos que possam embasar decisões de gestores públicos e privados do setor de transporte.

## 🎯 Problemática
**"Quais são os diferentes perfis operacionais das Regiões Metropolitanas brasileiras e como podemos agrupá-las para direcionar investimentos e políticas públicas de mobilidade?"**

## 1. Importação de Bibliotecas e Configurações

Nesta etapa, importamos as bibliotecas essenciais para análise de dados, visualização e modelagem de clusterização.

In [8]:
# 1. Importações e Configurações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
%matplotlib inline

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


## 2. Carregamento e Entendimento dos Dados

A base utilizada é o **PEMOB 2025** (Pesquisa de Mobilidade Urbana), que contém informações sobre infraestrutura, operação, demanda, finanças e tecnologia do transporte público nas Regiões Metropolitanas brasileiras.

A base possui um cabeçalho mesclado, por isso a leitura é feita a partir da segunda linha (`header=1`). Também realizamos a limpeza de valores textuais que representam nulos (`NA`, `NR`, `N/E`).

In [9]:
# 2. Carregando os dados brutos
caminho_base = '../data/raw/pemob_metropolitana_2025.xlsx'

df = pd.read_excel(caminho_base, header=1)
df = df.rename(columns={df.columns[0]: 'UF', df.columns[1]: 'Regiao_Metropolitana'})
df = df.replace(['NA', 'NR', 'N/E', 'Nenhum', ' '], np.nan)

print(f"✅ Base carregada: {df.shape[0]} Regiões Metropolitanas e {df.shape[1]} colunas.")
print("\n📋 Primeiras linhas da base:")
display(df[['UF', 'Regiao_Metropolitana']].head(10))

✅ Base carregada: 29 Regiões Metropolitanas e 344 colunas.

📋 Primeiras linhas da base:


,UF,Regiao_Metropolitana
0,AL,Região Metropolitana de Maceió
1,AM,Região Metropolitana de Manaus
2,BA,Região Metropolitana de Salvador
3,ES,Região Metropolitana da Grande Vitória
4,GO,Região Metropolitana de Goiânia
5,MG,Região Metropolitana de Belo Horizonte
6,PE,Região Metropolitana de Recife
7,PI,Região Integrada de Desenvolvimento da Grande ...
8,PR,Região Metropolitana de Curitiba
9,PR,Região Metropolitana de Londrina


## 3. Seleção de Variáveis de Interesse

A base original possui **344 colunas**, o que é inviável para algoritmos de clusterização (maldição da dimensionalidade). Selecionamos um subconjunto de **7 variáveis** que representam as três dimensões fundamentais da mobilidade:

| Dimensão | Variável | Descrição |
|----------|----------|-----------|
| **Infraestrutura** | `1.2.1.A` | Nº de terminais rodoviários |
| **Infraestrutura** | `1.3.1.A` | Frota de ônibus operacionais |
| **Operação** | `1.4.1` | Km percorrida por ônibus em 2024 |
| **Qualidade** | `2.1.1` | Idade média da frota de ônibus (anos) |
| **Demanda** | `3.2.1.A` | Nº de passageiros comuns |
| **Demanda** | `3.2.1.B` | Nº de passageiros com Vale Transporte |
| **Demanda** | `3.2.1.C` | Nº de estudantes transportados |

In [10]:
# 3. Selecionando as variáveis de interesse
variaveis_interesse = [
    '1.2.1.A',      # Nº de terminais rodoviários
    '1.3.1.A',      # Frota de ônibus operacionais
    '1.4.1',        # Km percorrida por ônibus em 2024
    '2.1.1',        # Idade média da frota de ônibus (anos)
    '3.2.1.A',      # Nº de passageiros comuns transportados
    '3.2.1.B',      # Nº de passageiros com Vale Transporte
    '3.2.1.C'       # Nº de estudantes transportados
]

df_cluster = df[['UF', 'Regiao_Metropolitana'] + variaveis_interesse].copy()

# Convertendo para numérico
for col in variaveis_interesse:
    df_cluster[col] = pd.to_numeric(df_cluster[col], errors='coerce')

# Removendo RMs com dados faltantes
df_cluster = df_cluster.dropna()

print(f"✅ Base final: {df_cluster.shape[0]} RMs e {df_cluster.shape[1]} colunas")
print("\n📊 Resumo estatístico das variáveis:")
display(df_cluster[variaveis_interesse].describe().round(2))

✅ Base final: 5 RMs e 9 colunas

📊 Resumo estatístico das variáveis:


,1.2.1.A,1.3.1.A,1.4.1,2.1.1,3.2.1.A,3.2.1.B,3.2.1.C
count,5.00,5.00,5.000000e+00,5.00,5.000000e+00,5.000000e+00,5.000000e+00
mean,14.20,1586.40,9.674595e+07,7.84,7.174876e+08,1.300361e+08,3.830763e+08
std,6.91,1291.49,7.336957e+07,1.16,7.427155e+08,1.419556e+08,3.215423e+08
min,5.00,324.00,2.639385e+07,7.00,1.507583e+08,1.852021e+06,1.051527e+07
25%,10.00,771.00,5.350724e+07,7.20,3.467098e+08,4.169863e+07,1.681562e+08
50%,16.00,1389.00,7.550423e+07,7.20,3.880179e+08,1.040285e+08,3.659399e+08
75%,17.00,1779.00,1.133082e+08,8.00,7.041831e+08,1.366613e+08,5.345697e+08
max,23.00,3669.00,2.150162e+08,9.80,1.997769e+09,3.659399e+08,8.362004e+08


## 4. Padronização das Variáveis (StandardScaler)

**Por que padronizar?**

Algoritmos baseados em distância (K-Means, Hierárquico) são **sensíveis à escala**. Observe no resumo estatístico:

- **Passageiros** estão na casa dos **milhões**
- **Idade da frota** está na casa das **dezenas**
- **Terminais** estão na casa das **unidades**

Sem padronização, a variável "Passageiros" dominaria completamente o cálculo da distância Euclidiana. Aplicamos então o **StandardScaler**, que transforma os dados para **média 0 e desvio padrão 1**.

In [11]:
# 4. Padronização
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster[variaveis_interesse])

df_scaled = pd.DataFrame(df_scaled, columns=variaveis_interesse)
df_scaled['Regiao_Metropolitana'] = df_cluster['Regiao_Metropolitana'].values
df_scaled['UF'] = df_cluster['UF'].values

print("✅ Dados padronizados!")
print("\n📊 Verificação (média ~0 e desvio padrão ~1):")
display(df_scaled[variaveis_interesse].describe().round(4))

✅ Dados padronizados!

📊 Verificação (média ~0 e desvio padrão ~1):


,1.2.1.A,1.3.1.A,1.4.1,2.1.1,3.2.1.A,3.2.1.B,3.2.1.C
count,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000
mean,0.0000,-0.0000,-0.0000,-0.0000,-0.0000,0.0000,0.0000
std,1.1180,1.1180,1.1180,1.1180,1.1180,1.1180,1.1180
min,-1.4893,-1.0928,-1.0721,-0.8089,-0.8531,-1.0096,-1.2954
25%,-0.6799,-0.7059,-0.6589,-0.6163,-0.5581,-0.6957,-0.7473
50%,0.2914,-0.1709,-0.3237,-0.6163,-0.4960,-0.2048,-0.0596
75%,0.4533,0.1667,0.2524,0.1541,-0.0200,0.0522,0.5268
max,1.4246,1.8029,1.8022,1.8874,1.9272,1.8580,1.5756
